# Task 4: Fine-Tuning BERT on Dataset

This project demonstrates fine-tuning a BERT transformer model for sentiment classification using Hugging Face Transformers.

In [1]:
!pip install transformers datasets torch accelerate -q

In [2]:
import pandas as pd

from datasets import Dataset

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.model_selection import train_test_split

In [3]:
data = {
    "text": [
        "I love this product",
        "This movie is amazing",
        "Worst experience ever",
        "I hate this service",
        "Absolutely fantastic",
        "Very disappointing",
        "Excellent work",
        "Terrible support",
        "I am very happy",
        "Not worth the money"
    ],

    "label": [1,1,0,0,1,0,1,0,1,0]
}

df = pd.DataFrame(data)

df

,text,label
0,I love this product,1
1,This movie is amazing,1
2,Worst experience ever,0
3,I hate this service,0
4,Absolutely fantastic,1
5,Very disappointing,0
6,Excellent work,1
7,Terrible support,0
8,I am very happy,1
9,Not worth the money,0


In [4]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42
)

In [5]:
tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [6]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True
)

In [7]:
train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": train_labels
})

test_dataset = Dataset.from_dict({
    "input_ids": test_encodings["input_ids"],
    "attention_mask": test_encodings["attention_mask"],
    "labels": test_labels
})

In [8]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    logging_dir="./logs",
    logging_steps=1
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [11]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,0.671494
2,0.664258
3,0.517029
4,0.543579


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4, training_loss=0.5990902334451675, metrics={'train_runtime': 21.028, 'train_samples_per_second': 0.38, 'train_steps_per_second': 0.19, 'total_flos': 24666661440.0, 'train_loss': 0.5990902334451675, 'epoch': 1.0})

In [12]:
sample_text = "I really love this product"

inputs = tokenizer(
    sample_text,
    return_tensors="pt"
)

outputs = model(**inputs)

prediction = outputs.logits.argmax().item()

if prediction == 1:
    print("Positive Sentiment")
else:
    print("Negative Sentiment")

Negative Sentiment


# Conclusion

In this project, we fine-tuned a BERT transformer model for sentiment classification using Hugging Face Transformers.

This task helped in understanding:
- BERT tokenization
- Transformer models
- Fine-tuning process
- NLP classification workflow